# NanoScale-LM — measure it, then use it

The first two notebooks build a model and compress it. This one asks the two questions that
actually decide whether it is worth anything:

1. **Is it good?** — measured with metrics that survive contact with a reviewer:
   bits-per-byte (comparable across tokenizers), a forced-choice grammar benchmark with
   confidence intervals, and calibration.
2. **What is it for?** — a language model is a compressor and an anomaly detector, and both
   fall out of the same forward pass.

Runs on CPU. Training the model takes ~10 minutes at the small tier used here; if you have
a checkpoint already, skip to section 2.

## 0. Setup

In [ ]:
!git clone --depth 1 https://github.com/vedant1711/nanoscale-lm 2>/dev/null || true
%cd nanoscale-lm
!pip install -q -e ".[dev]"

import torch

print("torch", torch.__version__)

## 1. Train something worth measuring

The `nano` tier on the synthetic corpus trains in ~95 seconds on CPU. It is enough to
demonstrate every technique below; the reported numbers in the repository come from the 40M
TinyStories model, which needs a few hours.

In [ ]:
from nanoscale.config import TokenizerConfig, load_experiment
from nanoscale.data.toy import generate_corpus
from nanoscale.tokenizer import BPETokenizer
from nanoscale.train import Trainer

tok = BPETokenizer.train(
    generate_corpus(seed=1337), TokenizerConfig(vocab_size=1024, max_train_bytes=1_200_000)
)
cfg = load_experiment(tier="nano", overrides=["train.device=cpu"])
trainer = Trainer(cfg, tokenizer=tok, out_dir="runs/nb3/pretrain")
trainer.train()
model = trainer.model

## 2. Bits per byte — the only likelihood metric that compares across models

Perplexity is per *token*, and token boundaries belong to the tokenizer, not to the text. A
model with a 50k vocabulary needs fewer tokens for the same sentence, so each token carries
more information and its perplexity looks **worse** — even when it models the text better.

Bits-per-byte normalises by the UTF-8 length of the source, which no tokenizer can change:

```
BPB = (Σ negative log-likelihood in nats / ln 2) / bytes of text
```

This is what the Pile and Chinchilla papers report for cross-model comparison, and it is
what makes `scripts/external_baseline.py` a fair test rather than an artefact of vocabulary
size.

In [ ]:
from nanoscale.eval import bits_per_byte, perplexity
from nanoscale.train import TokenBatcher, build_packed_tokens

data = build_packed_tokens(cfg.data, tok)
batches = TokenBatcher(data.val, seq_len=cfg.data.seq_len, batch_size=8, shuffle=False).take(16)

# The byte count must come from the *evaluated* tokens, not a global ratio.
eval_bytes = sum(
    len(tok.decode([int(t) for t in row if int(t) >= 0]).encode("utf-8"))
    for b in batches
    for row in b.targets
)

ppl = perplexity(model, batches)
bpb = bits_per_byte(model, batches, n_bytes=eval_bytes)
print(f"token perplexity : {ppl.perplexity:.4f}   <- depends on the tokenizer")
print(f"bits per byte    : {bpb.bits_per_byte:.4f}   <- does not")
print(f"bytes per token  : {bpb.bytes_per_token:.2f}")

## 3. A benchmark with headroom

The repository's first benchmark was 28 hand-written questions and the base model scored
100% on it — so it could detect *damage* from compression and could not measure capability
at all.

The replacement is BLiMP-style (Warstadt et al. 2020): each item is two sentences differing
in exactly one place, one grammatical and one not. The model scores both and is right if it
prefers the grammatical one. Chance is exactly 50%, and difficulty is controllable.

In [ ]:
from nanoscale.eval import generate_pairs, run_minimal_pairs

pairs = generate_pairs(n_per_phenomenon=60, seed=1337)
print(f"{len(pairs)} items\n")
for name in ["agreement_simple", "agreement_attractor", "negation"]:
    ex = next(p for p in pairs if p.phenomenon == name)
    print(f"{name}:\n  GOOD  {ex.good}\n  BAD   {ex.bad}\n")

In [ ]:
result = run_minimal_pairs(model, tok, pairs=pairs)

print(f"{'phenomenon':22} {'acc':>6}  {'95% CI':>14}   above chance?")
for s in result.scores:
    lo, hi = s.interval
    print(
        f"{s.phenomenon:22} {s.accuracy * 100:5.1f}%  "
        f"[{lo * 100:5.1f},{hi * 100:5.1f}]   {'yes' if s.above_chance else 'NO'}"
    )
print(f"\nmacro-average: {result.overall * 100:.1f}%  (chance = 50%)")

### Your numbers above are at chance. That is the correct result, and it is the point.

The `nano` model you just trained scores ~50% almost everywhere — indistinguishable from
guessing. Nothing is broken. The suite's lexicon (*ducks*, *reflexives*, *determiners*) comes
from ordinary English, and this model was trained on a narrow synthetic story grammar with a
1,024-token vocabulary. The probes are **structurally off-distribution**, so the model has no
opinion about them and the benchmark measures nothing.

This is worth dwelling on because the project hit exactly this bug for real. An early
agreement probe scored *precisely* 0.5, which read as "the model cannot do agreement". It
could: the probe was a single sentence and the model had only ever seen multi-sentence
stories. Rewriting the probe with a proper story prefix took the same model from 6/12 to
12/12.

**A benchmark returning exactly chance is more likely to be broken than informative**, and
"the model is bad" is the more comfortable of the two conclusions — which is why it needs
checking first.

The real numbers come from the 40M TinyStories model, which is in-distribution for this
suite:

| phenomenon | 40M TinyStories |
|---|---|
| reflexive | 100% |
| pronoun_gender | 100% |
| tense_consistency | 98% |
| agreement_simple | 94% |
| determiner_noun | 86% |
| entity_tracking | 76% |
| argument_structure | 64% |
| **negation** | **50%** — never learned |
| **agreement_attractor** | **44%** — below chance |

Reproduce with `python scripts/evaluate.py runs/micro/tinystories/final.pt`.

### The diagnostic pair to look at

Compare `agreement_simple` with `agreement_attractor`. They test the same rule; the second
inserts a noun of the *opposite* number between the subject and the verb:

```
The boy near the cats runs.     <- subject is "boy", singular
The boy near the cats run.      <- agrees with "cats" instead
```

A model that has learned syntactic agreement handles both. A model that has learned
"match the nearest noun" scores well on the first and **at or below chance** on the second.
The 40M TinyStories model scores 94% and 44% respectively — below chance, which is a
stronger result than at-chance: it means there *is* a systematic rule and it is the wrong
one.

## 4. Calibration — is the confidence earned?

A model can be accurate and badly calibrated: 99% confident on predictions that are right
70% of the time. That matters here specifically because over-confidence is the mechanism
behind degenerate generation — a model certain of a wrong continuation cannot recover.

In [ ]:
from nanoscale.eval import calibration

cal = calibration(model, batches)
print(f"top-1 accuracy    : {cal.accuracy * 100:.1f}%")
print(f"mean confidence   : {cal.mean_confidence * 100:.1f}%")
print(f"over-confidence   : {cal.overconfidence:+.4f}   (positive = too sure of itself)")
print(f"expected cal. err : {cal.ece:.4f}")

## 5. A language model is a compressor

Shannon: a symbol of probability `p` costs `-log2(p)` bits. So the cross-entropy measured in
section 2 *is* a file size — feed the model's next-token distribution to an arithmetic coder
and you get a real, smaller file back.

The repository implements the coder, so this is measured rather than asserted.

In [ ]:
import bz2
import gzip
import lzma

from nanoscale.compress import compress, decompress

text = (
    "It was a sunny day. Lily went to the park with a small dog. "
    "She wanted to find a red ball. But the ball was stuck under a bench. "
    "She asked her friend for help and together they pulled it free."
)
raw = text.encode("utf-8")

r = compress(model, tok, text)
restored = decompress(model, tok, r.payload, r.n_tokens)
assert restored == text, "not lossless!"

print(f"{'raw UTF-8':12} {len(raw):5d} B   8.000 bits/byte")
for name, blob in [
    ("gzip -9", gzip.compress(raw, 9)),
    ("bzip2 -9", bz2.compress(raw, 9)),
    ("xz -9", lzma.compress(raw, preset=9)),
]:
    print(f"{name:12} {len(blob):5d} B   {len(blob) * 8 / len(raw):.3f} bits/byte")
print(
    f"{'NanoScale':12} {r.n_bytes_out:5d} B   {r.bits_per_byte:.3f} bits/byte "
    f"  <- lossless round trip verified"
)
print(f"\ncoder overhead above the model's own cross-entropy: {r.coder_overhead * 100:.2f}%")
print(
    "\nNote: on a passage this short xz can come out *larger* than the input — its "
    "container and dictionary cost more than it saves. The neural coder has a 4-byte "
    "flush and no header, so it stays honest at small sizes."
)

### Now try text the model has never seen

This is where the argument lives. Run the cell below on out-of-domain prose and watch the
advantage invert.

In [ ]:
out_of_domain = (
    "The mitochondrion is a double-membrane-bound organelle found in most eukaryotic "
    "organisms, generating most of the cell's supply of adenosine triphosphate."
)
r2 = compress(model, tok, out_of_domain)
raw2 = out_of_domain.encode("utf-8")
print(f"in domain     : {r.bits_per_byte:.3f} bits/byte  ({r.ratio:.2f}x)")
print(f"out of domain : {r2.bits_per_byte:.3f} bits/byte  ({r2.ratio:.2f}x)")
print(f"xz on the same: {len(lzma.compress(raw2, preset=9)) * 8 / len(raw2):.3f} bits/byte")

**That collapse is the design, not a defect.** The model is not a general compressor; it is
a compressor *for one distribution*, and it buys its in-domain advantage by being useless
elsewhere. Which is exactly the trade you want when the thing being archived is a billion
log lines that all look alike.

The economics decide whether it is worth doing at all — the model ships with the archive, so
there is a break-even volume. At int4 the 40M model is 22 MB and breaks even at ~99 MB of
in-domain text, which any log pipeline exceeds in a day. `scripts/compression_bench.py`
works this out in full.

## 6. The same forward pass is an anomaly detector

Per-token surprisal is the compressor's cost function, un-summed. A line the model finds
expensive to encode is a line unlike its training distribution — an unsupervised anomaly
score with no labels and one threshold.

In [ ]:
from nanoscale.compress import score_lines, token_surprisal

lines = [
    "It was a sunny day and Lily went to the park.",
    "Tom found a red ball under the old bench.",
    "The dog was very happy to see its friend.",
    "Tom picked up the quantum entanglement and put it in his pocket.",
    "SELECT * FROM users WHERE id = 42 ORDER BY created_at DESC;",
    "xqzk vburt plimf woggle zzzt krrn.",
]
report = score_lines(model, tok, lines)
floor = min(report.scores)
print(f"{'bits/tok':>9} {'vs best':>8}   line")
for score, line in report.ranked():
    flag = "  <-- anomalous" if score > floor * 1.5 else ""
    print(f"{score:9.2f} {score / floor:7.1f}x   {line[:52]}{flag}")

In [ ]:
# Per-token view: where exactly does the cost land?
for s in [
    "Tom found a red ball under the old bench.",
    "Tom picked up the quantum entanglement and put it in his pocket.",
]:
    print(s)
    for token, bits in token_surprisal(model, tok, s):
        bar = "#" * min(24, int(bits * 2))
        print(f"  {token!r:>14} {bits:6.2f} {bar}")
    print()

The interesting row is never the gibberish. Look at the *quantum entanglement* sentence: it
is grammatical, in the right register, with the right characters — and it costs several
times what an ordinary sentence costs, because one noun phrase is impossible in this world.
That is semantic detection, not spell-checking, and it needs no labels at all.

## Where to go next

- `scripts/evaluate.py` — the full report on any checkpoint
- `scripts/external_baseline.py` — bits-per-byte against GPT-2 and distilGPT-2
- `scripts/emergence.py` — probe the benchmark *through* training, not just at the end
- `scripts/compression_bench.py` — the codec benchmark and the break-even economics
- [The documentation](https://vedant1711.github.io/nanoscale-lm/explainer.html) — all of it
  in one page